# Agentic Pipeline Design for scRNA-seq Re-analysis

This notebook documents the architecture for automating the re-analysis workflow developed
across NB8–NB15. It is written as a reference for building the agentic system, not as a
runnable pipeline — code cells here are design sketches, not production code.

---

## The workflow this automates

The NB8–NB15 analysis followed a specific pattern that's generalizable to any published
scRNA-seq dataset:

**Step 0: Identify a gap in an existing study.**
Yang et al. 2022 used Wilcoxon tests, treating cells as independent replicates. Pseudobulk
DESeq2 (treating mice as replicates) is more statistically correct. That gap — a better
method exists but wasn't used — was the entry point. This could also be: the study didn't
examine a specific cell type, didn't run CCC analysis, didn't look at a specific gene family.

**Step 1: Run the better method. Find significant hits.**
Run DESeq2. Get a ranked list of significant genes × cell states.

**Step 2: Credibility-check each hit.**
Three data-execution checks that rule out artifacts before investing in literature search:
- Per-mouse rank separation (not a single-outlier artifact)
- Cell-state specificity (not 20 cell states all moving together)
- Cross-tissue direction (contamination predicts same direction; biology often predicts opposite)

**Step 3: Literature-check credible hits for novelty.**
Query at cell-type resolution, not gene resolution. Validate that citations aren't the
source dataset itself. Check whether the study's own paper already reported the finding.

**Step 4: Generate downstream hypotheses from credible + novel hits.**
If irisin is implicated: check the receptor. Check co-expression in single cells. Check
downstream targets from literature. This step is LLM reasoning on accumulated context —
not hardcoded biology.

**Step 5: Confirm hypotheses with data.**
Run the proposed analyses. Write results back to context.

**Step 6: Write the narrative.**
Build the document from the accumulated findings, evidence strength, and caveats.

---

## What makes this different from generic deep research

A generic deep research agent does: `query → web_search → synthesize → report`.
Its claims are grounded by citations.

This pipeline does: `data → execute → credibility_check(data) → novelty_check(web) → confirm(data)`.
Its claims are grounded by numbers produced by code on the actual dataset.

The two `→ data →` steps are what a generic agent cannot do. They require:
1. Access to the dataset as a tool, not just as context
2. Knowledge of what checks are methodologically appropriate (per-mouse, cross-tissue)
3. The ability to interpret numerical results in domain context

The domain knowledge doesn't need to be hardcoded. It comes from the LLM reading the
accumulated context document — which grows as the analysis progresses, exactly like
PIPELINE.md did in this project.

---

## The persistent context document

PIPELINE.md was the most important artifact in the NB8–NB15 workflow. It was:
- A living record of what was found and why it was (or wasn't) credible
- The domain knowledge store — each agent read it to understand what had already been tried
- The framing document — it recorded not just findings but how to present them

In an automated pipeline, this becomes a structured document that agents write to
after each step and read at the start of each step. Its format:

In [ ]:
# Sketch: the persistent context document structure
# This is what every agent reads at startup and writes to on completion.

CONTEXT_SCHEMA = """
# Analysis Context

## Dataset
- Source paper: {citation}
- GEO accession: {geo_id}
- Shape: {n_cells} cells × {n_genes} genes
- Tissues: {tissues}
- Conditions: {conditions}
- Cell states: {cell_states}
- Analysis gap: {gap_identified}  ← WHY we're re-analyzing; what the paper didn't do

## Findings
<!-- One entry per credible finding. Status: candidate|credible|novel|confirmed|ruled_out -->

### Finding: {gene} in {cell_state} ({contrast})
- Status: {status}
- Statistics: log2FC={log2fc}, padj={padj}
- Credibility: {credibility_summary}
- Novelty: {novelty_summary}
- Downstream checks: {downstream_checks}
- Current framing: {framing}

## Ruled Out
<!-- Findings that failed credibility or were not novel; brief reason why -->
- {gene}: {reason}

## Open Questions
<!-- Generated by Hypothesis Agent; consumed by Confirmation Agent -->
- [ ] {question}
"""

print("Context schema defined. This document grows throughout the pipeline.")
print("Every agent reads it at startup; every agent writes to it on completion.")

---

## The data interface

Every agent that touches the dataset needs a consistent interface for executing analyses
and returning structured results. The key design decision: results must be **numbers +
interpretation**, not just numbers. The agent writes the interpretation to context;
the next agent builds on it.

In [ ]:
# Sketch: the Finding dataclass that flows between agents
from dataclasses import dataclass, field
from typing import Literal

@dataclass
class CredibilityResult:
    passed: bool
    rank_separation: bool      # min(group_a) > max(group_b)?
    cell_state_specific: bool  # or is the same gene hitting 10+ states?
    cross_tissue_direction: str  # 'opposite' | 'same' | 'absent' | 'inconclusive'
    notes: str

@dataclass
class NoveltyResult:
    is_novel: bool
    confidence: float          # 0–1; low if literature is sparse or ambiguous
    prior_art: list[str]       # citations that ARE prior art (cell-type specific)
    false_positives: list[str] # citations the agent rejected (circular, wrong granularity)
    notes: str

@dataclass
class AnalysisProposal:
    description: str           # plain English: what to check and why
    code_template: str         # Python code to execute
    expected_outcome: str      # what would strengthen vs. weaken the finding
    priority: Literal['high', 'medium', 'low']

@dataclass
class Finding:
    gene: str
    cell_state: str
    tissue: str
    contrast: str
    log2fc: float
    padj: float
    
    credibility: CredibilityResult | None = None
    novelty: NoveltyResult | None = None
    downstream: list['Finding'] = field(default_factory=list)
    proposals: list[AnalysisProposal] = field(default_factory=list)
    framing: str = ''          # how to describe this finding; updates as context grows
    status: Literal[
        'candidate', 'credible', 'novel', 'confirmed', 'ruled_out'
    ] = 'candidate'

print("Finding dataclass defined.")

---

## Agent 0: Gap Identifier

**Input:** The source paper (PDF or abstract) + the h5ad file metadata.

**What it does:** Identifies the gap between what the paper did and what's possible
with the data. This is the entry point that makes this a *re-analysis* pipeline rather
than a generic analysis pipeline.

**How it works in practice:** The LLM reads the paper's methods section and compares
against a checklist of modern scRNA-seq best practices. Gaps become the `analysis_gap`
field in the context document.

**Examples of gaps this would find:**
- Paper used Wilcoxon on cells → pseudobulk DESeq2 is better (what we did)
- Paper did bulk DEG but no CCC analysis → apply LIANA or manual LR scoring
- Paper clustered coarsely → re-cluster at higher resolution to find sub-states
- Paper looked at one tissue → cross-tissue comparison was never done
- Paper did not check a specific gene family of interest

In [ ]:
# Sketch: Gap Identifier system prompt + tool use

GAP_IDENTIFIER_SYSTEM = """
You are analyzing a published single-cell RNA-seq study to identify methodological
gaps or unexplored angles that could be addressed by re-analyzing the public dataset.

Check against this list of modern scRNA-seq best practices:
1. Statistical testing: Did they use pseudobulk methods (DESeq2, edgeR) treating mice as
   replicates? Or Wilcoxon/t-test on individual cells (inflated N, false positives)?
2. Cell-cell communication: Did they run CCC analysis (LIANA, CellPhoneDB, or manual LR
   scoring)? If not, which ligand-receptor pairs are biologically motivated?
3. Trajectory analysis: Did they model differentiation trajectory (Monocle, DPT)?
4. Cross-tissue comparison: Did they compare the same cell type across all tissues?
5. Rescue contrast: Did they run the TH vs SH comparison (exercise in obese animals),
   or only the simpler obesity and exercise contrasts?
6. Cell-state resolution: Did they annotate sub-states within major cell types, or only
   broad types?
7. Literature connection: Are there recent papers (last 2 years) that establish a
   mechanism relevant to genes in this dataset that the paper couldn't have cited?

For each gap found, output:
- Gap description (1 sentence)
- Why the better method produces a qualitatively different result, not just a marginal one
- Estimated analysis complexity: simple (1 notebook section) | medium (full notebook) | hard
- What finding type this gap could produce: DEG | CCC | trajectory | cross-tissue | other
"""

# In this project:
gap_example = {
    'gap': 'Paper used Wilcoxon tests treating cells as replicates',
    'better_method': 'Pseudobulk DESeq2 treating mice as replicates',
    'qualitative_difference': (
        'Wilcoxon finds hundreds of DEGs per cell state by inflating N to ~1000s of cells. '
        'DESeq2 finds 13–167 with N=3 mice. The DESeq2 results are defensible; '
        'the Wilcoxon results are not publishable.'
    ),
    'complexity': 'medium',
    'finding_type': 'DEG',
}
print("Gap example (what we found in Yang et al. 2022):")
for k, v in gap_example.items():
    print(f"  {k}: {v}")

---

## Agent 1: Scanner

**Input:** h5ad file + analysis gap from context document.

**What it does:** Runs the analysis that addresses the gap. For the DEG gap, this is
pseudobulk DESeq2 across all cell states × all contrasts. Returns a ranked list of
`Finding` candidates sorted by credibility potential.

**Credibility potential** is not just padj. Low-count cell states with N=2 mice per
group are deprioritized even if padj is low. The ranking considers: n_cells per mouse,
n_mice per group, effect size, and whether the gene is in a relevant pathway.

In [ ]:
# Sketch: Scanner agent — what the actual scan looks like on our data
# This is the output from NB8 that seeds the rest of the pipeline.

import pandas as pd
from pathlib import Path

deg = pd.read_csv(Path('outputs/degs_vwat_cell_state_deseq2_pseudobulk.csv'))

# Credibility potential score: prefer large n_cells, large effect, not too many
# other significant genes in same cell state (specificity)
sig = deg[deg['is_sig'] == True].copy()

# How many sig genes per cell state? High count → less specific, lower priority
n_sig_per_state = sig.groupby('cell_state')['gene'].count().rename('n_sig_in_state')
sig = sig.join(n_sig_per_state, on='cell_state')

# Credibility potential: large effect + low n_sig_in_state = more specific signal
sig['credibility_potential'] = sig['log2FC'].abs() / (sig['n_sig_in_state'] ** 0.5)

top = (
    sig[sig['contrast'] == 'TC_vs_SC']
    .sort_values('credibility_potential', ascending=False)
    .head(10)
    [['gene', 'cell_state', 'log2FC', 'padj', 'n_sig_in_state', 'credibility_potential']]
)

print("Top 10 exercise hits by credibility potential (TC vs SC):")
print(top.round(3).to_string(index=False))
print()
print("Note: Fndc5/Areg ranks high because Areg has few significant exercise genes (30)")
print("compared to WAT_IPC (13 — but those are circadian, expected from paper).")

---

## Agent 2: Credibility Checker

**Input:** A `Finding` candidate from the Scanner.

**What it does:** Three independent data-execution checks. These are the checks that
distinguish this pipeline from a generic research agent — they require running code
on the actual dataset, not searching the web.

**All three checks run in parallel.** They're independent of each other.

**The three checks:**

1. **Per-mouse rank separation:** Sum counts per mouse in the two comparison groups.
   Does min(group_b) > max(group_a)? Perfect separation = strong credibility.
   One overlap = acceptable. Multiple overlaps = weak.

2. **Cell-state specificity:** How many other cell states show the same gene
   moving in the same direction? If Fndc5 goes up in Areg *and* WAT_IPC *and* CP
   *and* Fibroblast, it's a general stress response, not an Areg-specific finding.

3. **Cross-tissue direction:** Does the same gene move in the same or opposite
   direction in the same cell type (or its equivalent) in other tissues?
   - Same direction: probably a general response, less specific
   - Opposite direction: rules out contamination, more specific
   - Absent in other tissues: neutral (may be tissue-specific biology)

In [ ]:
# Sketch: Credibility Agent — the three checks as runnable functions
import numpy as np
import scanpy as sc
from scipy.sparse import issparse

def check_rank_separation(adata, gene, cell_state, tissue, cond_a, cond_b):
    """Check that every mouse in cond_b is above every mouse in cond_a."""
    sub = adata[(adata.obs['tissue'] == tissue) &
                (adata.obs['cell_state_label'] == cell_state)]
    
    per_mouse = {}
    for cond in [cond_a, cond_b]:
        group = sub[sub.obs['intervention_group'] == cond]
        per_mouse[cond] = {}
        for sid in group.obs['sample_ID'].unique():
            mouse = group[group.obs['sample_ID'] == sid]
            vals = mouse[:, gene].layers['counts']
            if issparse(vals): vals = vals.toarray().ravel()
            per_mouse[cond][sid] = float(vals.sum())
    
    min_b = min(per_mouse[cond_b].values())
    max_a = max(per_mouse[cond_a].values())
    n_overlaps = sum(1 for v_a in per_mouse[cond_a].values()
                     for v_b in per_mouse[cond_b].values() if v_b < v_a)
    
    return {
        'passed': min_b > max_a,
        'min_b': min_b, 'max_a': max_a, 'n_overlaps': n_overlaps,
        'per_mouse': per_mouse
    }


def check_cell_state_specificity(deg_df, gene, contrast, n_states_threshold=3):
    """Is this gene significant in many cell states, or just this one?"""
    sig_states = deg_df[
        (deg_df['gene'] == gene) &
        (deg_df['contrast'] == contrast) &
        (deg_df['is_sig'] == True)
    ]['cell_state'].tolist()
    
    return {
        'passed': len(sig_states) <= n_states_threshold,
        'n_significant_states': len(sig_states),
        'states': sig_states
    }


def check_cross_tissue_direction(adata, gene, cell_state, tissue_primary,
                                  cond_a, cond_b,
                                  tissue_equivalents: dict):
    """
    Compare direction in primary tissue vs equivalent cell states in other tissues.
    tissue_equivalents: {other_tissue: equivalent_cell_state}
    e.g. {'SM': 'FAP_Areg'} for the Areg → FAP_Areg equivalence
    """
    results = {}
    for other_tissue, other_state in tissue_equivalents.items():
        sub = adata[(adata.obs['tissue'] == other_tissue) &
                    (adata.obs['cell_state_label'] == other_state)]
        if sub.shape[0] < 20:
            results[other_tissue] = 'too_sparse'
            continue
        means = {}
        for cond in [cond_a, cond_b]:
            vals = sub[sub.obs['intervention_group'] == cond, gene].X
            if issparse(vals): vals = vals.toarray().ravel()
            means[cond] = float(vals.mean())
        delta = means[cond_b] - means[cond_a]
        results[other_tissue] = {
            'delta': round(delta, 4),
            'direction': 'up' if delta > 0.005 else ('down' if delta < -0.005 else 'flat'),
            'cell_state': other_state
        }
    return results


print("Three credibility check functions defined.")
print("Each returns a dict that gets written to the Finding's credibility field.")

In [ ]:
# Worked example: run the three checks on Fndc5/Areg — the actual finding from NB11

adata = sc.read_h5ad('../GSE183288_Single_cell_atlas.h5ad')
deg   = pd.read_csv(Path('outputs/degs_vwat_cell_state_deseq2_pseudobulk.csv'))

print("=== Check 1: Per-mouse rank separation ===")
r1 = check_rank_separation(adata, 'Fndc5', 'Areg', 'vWAT', 'SC', 'TC')
print(f"  Passed: {r1['passed']}")
print(f"  min(TC) = {r1['min_b']:.0f}  >  max(SC) = {r1['max_a']:.0f}: {r1['min_b'] > r1['max_a']}")
print(f"  Per-mouse counts: {r1['per_mouse']}")

print()
print("=== Check 2: Cell-state specificity ===")
r2 = check_cell_state_specificity(deg, 'Fndc5', 'TC_vs_SC')
print(f"  Passed: {r2['passed']}")
print(f"  Significant in {r2['n_significant_states']} cell state(s): {r2['states']}")

print()
print("=== Check 3: Cross-tissue direction ===")
r3 = check_cross_tissue_direction(
    adata, 'Fndc5', 'Areg', 'vWAT', 'SC', 'TC',
    tissue_equivalents={'SM': 'FAP_Areg', 'scWAT': 'Areg'}
)
for tissue, result in r3.items():
    print(f"  {tissue}: {result}")

print()
# Interpret: vWAT goes up, SM is flat, scWAT is too sparse
# Verdict: passes all three checks → credible finding
print("Verdict: CREDIBLE")
print("  Rank separation: perfect (min TC > max SC)")
print("  Specificity: 1 significant cell state")
print("  Cross-tissue: SM flat (rules out contamination); scWAT inconclusive")

---

## Agent 3: Novelty Checker

**Input:** A `Finding` that passed credibility.

**What it does:** Web search to check whether this specific finding has been reported.
The query must be at **cell-type resolution** — not "is Fndc5 in adipose known?" but
"is exercise-induced Fndc5 upregulation in CD142+ adipose stem cells known?"

**Two known failure modes to guard against:**

1. **Circular citation:** The source dataset's own paper gets cited as prior art.
   Detect this by checking if the citation DOI/title matches the source paper.
   We hit this with Gemini citing Yang et al. 2022 as evidence against our own finding.

2. **Resolution mismatch:** A bulk tissue study gets cited as prior art for a
   cell-type-specific finding. "Fndc5 in bulk adipose tissue" is not prior art for
   "Fndc5 in Areg cells specifically." The cell-type level is genuinely new even if
   the tissue level was known.

In [ ]:
# Sketch: Novelty Agent query construction

def build_novelty_query(finding: 'Finding', source_paper_doi: str) -> dict:
    """
    Build a structured prompt for the novelty check.
    Returns: {query_string, validation_rules}
    """
    query = (
        f"Literature search: Has the following specific finding been reported previously? \n\n"
        f"Finding: {finding.gene} is significantly upregulated in {finding.cell_state} cells "
        f"in {finding.tissue} tissue with exercise training ({finding.contrast}). "
        f"log2FC={finding.log2fc:.2f}, padj={finding.padj:.4f}.\n\n"
        f"Search requirements:\n"
        f"1. Citations must be at cell-type resolution ({finding.cell_state}), not bulk tissue.\n"
        f"2. Do NOT cite {source_paper_doi} — that is the source dataset we are re-analyzing.\n"
        f"3. If only bulk-tissue evidence exists, report as PARTIAL NOVELTY, not prior art.\n"
        f"4. Check whether {finding.gene}'s known biology is consistent with "
        f"expression in {finding.cell_state} (e.g. is it known as muscle-specific?).\n\n"
        f"Output: is_novel (yes/no/partial), confidence (0-1), prior_art (list), "
        f"rejected_citations (list with reason), notes."
    )
    return {
        'query': query,
        'source_doi': source_paper_doi,
        'validation_rules': [
            'reject_if_is_source_paper',
            'reject_if_bulk_only_for_cell_type_finding',
            'flag_if_gene_known_tissue_specific',
        ]
    }


# What this produced for our Fndc5 finding:
fndc5_finding_sketch = type('Finding', (), {
    'gene': 'Fndc5', 'cell_state': 'Areg (CD142+)', 'tissue': 'vWAT',
    'contrast': 'TC_vs_SC', 'log2fc': 1.50, 'padj': 0.008
})()

q = build_novelty_query(fndc5_finding_sketch, '10.1016/j.cmet.2022.09.005')
print(q['query'])
print()
print("Validation rules that would have caught the Gemini error:")
print("  reject_if_is_source_paper → would have caught Yang et al. 2022 citation")
print("  reject_if_bulk_only_for_cell_type_finding → would have caught Moreno-Navarrete")
print("    2013 (bulk adipose) being cited as prior art for Areg-specific finding")

---

## Agent 4: Hypothesis Generator

**Input:** A `Finding` that is credible + novel + the full context document so far.

**What it does:** Proposes downstream analyses that could strengthen or refute the
finding. This is the LLM doing domain reasoning on accumulated context — not hardcoded
biology. The key insight: the domain knowledge comes from the context document, which
includes what was found, what was ruled out, and what the literature says.

**The system prompt provides methodology scaffolding** (not biology):
- For any significant DEG: check if receptor is expressed in the same cell type
- For any ligand finding: check if the ligand is cleaved by available proteases
- For any co-upregulation at group level: verify single-cell co-expression
- For any cell-type finding: check the equivalent cell type in other tissues
- For any finding with a known downstream pathway: look for pathway targets in the DEG list

These are **structural rules** a statistician could write. The LLM applies them to the
specific biology by reading the context document.

In [ ]:
# Sketch: Hypothesis Agent system prompt

HYPOTHESIS_AGENT_SYSTEM = """
You are designing follow-up analyses for a single-cell RNA-seq re-analysis.
You have been given:
  1. A credible, novel finding (gene × cell_state × contrast)
  2. The full context document showing what has already been analyzed
  3. A biology summary for the dataset (tissues, conditions, cell states, research question)

Apply these structural rules to generate analysis proposals:

RULE 1 — Receptor check:
  If the finding is a ligand or signaling molecule, check whether its receptor is
  expressed in the same or adjacent cell types. Use the known receptor from literature
  or propose a web search to find it.

RULE 2 — Cleavage machinery:
  If the finding involves a secreted precursor (e.g. a pro-form that must be cleaved),
  check whether the relevant protease (ADAM family, PCSK9, furin, etc.) is expressed
  constitutively in the same cell type.

RULE 3 — Single-cell co-expression:
  If two genes co-upregulate at the group mean level, verify they co-express in the
  same individual cells. Compute double-positive fraction SC vs TC.

RULE 4 — Cross-tissue equivalence:
  If a finding is in cell state X in tissue A, check the equivalent cell state in
  tissue B (use the atlas cell state annotations to identify the equivalent).

RULE 5 — Downstream pathway targets:
  If the literature check identified a known downstream pathway, search the DEG list
  for pathway target genes. Are any significantly up or down in the expected direction?

RULE 6 — Bi-directional regulation:
  If a gene goes up with exercise in cell state X, check whether it also changes with
  obesity and/or rescue in cell state X or related states.

RULE 7 — Honest negative check:
  For every mechanistic prediction the finding implies, explicitly check whether the
  data confirms or contradicts it. Report contradictions as findings, not failures.

For each proposal, output:
  - description: what to check and why (1-2 sentences)
  - code_template: the Python code to run (use the provided data interface)
  - expected_outcome: what would strengthen vs. weaken the finding
  - priority: high | medium | low
"""

# What this produced for our Fndc5 finding (as actually run in NB11-14):
proposals_generated = [
    {
        'description': 'Check Itgb5 (irisin receptor) expression in same cell types',
        'rule': 'RULE 1 — Receptor check',
        'result': 'Itgb5 mean 0.52 Areg, 0.54 WAT_IPC — top-2 in vWAT. Passes.',
        'notebook': 'NB11 Section 3'
    },
    {
        'description': 'Check Adam10/17 protease expression in Areg cells',
        'rule': 'RULE 2 — Cleavage machinery',
        'result': 'Adam10=0.10, Adam17=0.13 constitutive. Passes.',
        'notebook': 'NB12 Section 1'
    },
    {
        'description': 'Check FAP_Areg in SM (muscle MSC equivalent)',
        'rule': 'RULE 4 — Cross-tissue equivalence',
        'result': 'FAP_Areg SC=0.017, TC=0.018. Flat. Sharpens vWAT specificity.',
        'notebook': 'NB11 Section 5'
    },
    {
        'description': 'Check Fndc5 in obesity (SH vs SC) and rescue (TH vs SH) contrasts',
        'rule': 'RULE 6 — Bi-directional regulation',
        'result': 'CP: down in obesity (padj=0.003), up in rescue (padj=0.004). Novel.',
        'notebook': 'NB11 Section 2'
    },
    {
        'description': 'Lit check returned Mu et al. 2026 irisin→IL-33→Treg. Check Il33 co-expression.',
        'rule': 'RULE 5 — Downstream pathway targets',
        'result': '5× double-positive increase SC→TC. Il33 trend (padj=0.548). Moderate.',
        'notebook': 'NB14'
    },
    {
        'description': 'Mu et al. predicts Treg expansion. Explicitly check Treg proportions.',
        'rule': 'RULE 7 — Honest negative check',
        'result': 'Tregs DECREASE with exercise. Reported honestly as unexpected finding.',
        'notebook': 'NB14 Section 5'
    },
]

print("Proposals generated by Hypothesis Agent for Fndc5/Areg finding:")
for p in proposals_generated:
    print(f"  [{p['rule'].split('—')[0].strip()}] {p['description']}")
    print(f"    → {p['result']}")

---

## Agent 5: Confirmation Agent

**Input:** An `AnalysisProposal` from the Hypothesis Agent.

**What it does:** Executes the proposed code, interprets the result, updates the
finding's status, and writes back to the context document.

**The key output is not just numbers** — it's structured interpretation:
- Does this result strengthen or weaken the finding?
- Is the result expected, unexpected, or inconclusive?
- What's the updated framing given this result?

The interpretation is what gets written back to the context document and becomes input
for the next round of hypothesis generation.

In [ ]:
# Sketch: Confirmation Agent interpretation schema

CONFIRMATION_INTERPRETATION_SCHEMA = """
You have run an analysis proposed by the Hypothesis Agent. Interpret the result.

Analysis: {description}
Numerical result: {result}
Expected outcome if finding is real: {expected_outcome}

Output:
  verdict: strengthens | weakens | inconclusive | unexpected
  explanation: 1-2 sentences describing what the number means for the finding
  updated_framing: if the framing of the main finding should change, how?
  new_questions: any new questions raised by this result (feeds back to Hypothesis Agent)
"""

# Worked example: the Treg result
treg_interpretation = {
    'analysis': 'Check Treg proportions with exercise (Mu et al. predicts expansion)',
    'numerical_result': 'TC: 0.78% vs SC: 1.58% — Tregs decrease with exercise',
    'expected_if_real': 'Tregs increase with exercise if irisin→IL-33→Treg circuit active',
    'verdict': 'unexpected',
    'explanation': (
        'Tregs decrease with lean exercise. Consistent with reduced inflammatory load '
        '(fewer firefighters when no fire). Inconsistent with naive Mu et al. prediction '
        'but that was in obese+pharmacological irisin context, not lean exercise.'
    ),
    'updated_framing': (
        'Do not claim Treg expansion as a confirmed consequence. '
        'Reframe: the Il33→Treg axis may be most relevant in the obese/therapeutic '
        'context. The lean exercise finding is the Fndc5+Il33 co-upregulation; '
        'the Treg story is CP rescue (padj=7.2e-7).'
    ),
    'new_questions': [
        'Does Treg proportion increase with obesity (SH vs SC)? If yes, '
        'more firefighters when there IS fire — supports the "fewer firefighters" interpretation.',
        'Is Il1rl1 (ST2) stably expressed on Tregs across conditions, '
        'confirming receiver cells are primed even if not expanding?'
    ]
}

print("Confirmation Agent interpretation for Treg result:")
for k, v in treg_interpretation.items():
    if isinstance(v, list):
        print(f"  {k}:")
        for item in v: print(f"    - {item}")
    else:
        print(f"  {k}: {v}")

---

## Agent 6: Narrative Writer

**Input:** All findings in final status + the full context document.

**What it does:** Writes the narrative notebook (`15_narrative.ipynb`) from the
accumulated findings, evidence strength ratings, and caveats.

**The structure follows the evidence hierarchy:**
- Lead with the strongest finding (most statistically robust + most specific)
- For each finding: problem first, then method, then result, then credibility evidence
- Unexpected results reported honestly as findings, not buried
- Every claim linked to a specific number from code execution

In [ ]:
# Sketch: the orchestrator that connects all agents
# This is the top-level pipeline; agents run in the order shown.

import asyncio

PIPELINE_PSEUDOCODE = """
async def run_pipeline(h5ad_path, paper_pdf_path, output_dir):

    # Initialize context document
    ctx = ContextDocument(dataset_path=h5ad_path)
    atlas = load_atlas(h5ad_path)

    # ── PHASE 0: Identify the gap ──────────────────────────────────
    gap = await gap_identifier_agent.run(
        paper=paper_pdf_path,
        atlas_metadata=atlas.obs.describe()
    )
    ctx.write('analysis_gap', gap)

    # ── PHASE 1: Scan ──────────────────────────────────────────────
    candidates = await scanner_agent.run(atlas, gap)
    ctx.write('candidates', candidates)                # ranked list of Findings

    # ── PHASE 2: Credibility (parallelizable per finding) ──────────
    credible = await asyncio.gather(*[
        credibility_agent.run(f, atlas)
        for f in candidates if f.padj < 0.05
    ])
    credible = [f for f in credible if f.credibility.passed]
    ctx.write('credible_findings', credible)

    # ── PHASE 3: Novelty (parallelizable) ─────────────────────────
    novel = await asyncio.gather(*[
        novelty_agent.run(f, source_doi=ctx.source_doi)
        for f in credible
    ])
    novel = [f for f in novel if f.novelty.is_novel]
    ctx.write('novel_findings', novel)

    # ── PHASE 4: Hypothesis → Confirmation (sequential per finding)
    #    Sequential because findings can inform hypotheses for later findings
    confirmed = []
    for finding in novel:
        proposals = await hypothesis_agent.run(
            finding,
            context=ctx.read_all()       # ← accumulated context matters here
        )
        for proposal in proposals:
            result = await confirmation_agent.run(proposal, atlas)
            finding.downstream.append(result)
            ctx.write('confirmation', result)   # ← feeds next hypothesis round
        confirmed.append(finding)

    # ── APPROVAL GATE ──────────────────────────────────────────────
    # Write checkpoint; human reviews before narrative is written
    ctx.write_checkpoint('pre_narrative_review')
    await human_approval_gate(ctx)       # async; pipeline pauses here

    # ── PHASE 5: Narrative ─────────────────────────────────────────
    notebook = await narrative_agent.run(
        findings=confirmed,
        context=ctx.read_all()
    )
    notebook.save(output_dir / '15_narrative.ipynb')
    return notebook
"""

print(PIPELINE_PSEUDOCODE)

---

## The three approval gates

A fully automated run would fail at three points where human judgment was load-bearing
in the NB8–NB15 workflow. These should be **async approval gates** — the pipeline
writes a checkpoint, notifies, and waits. Not blocking synchronous calls.

**Gate 1: After Novelty Check**
The agent reports which findings passed credibility + novelty. Human confirms:
- Is this actually interesting, or just statistically significant? (Fndc5 > Mef2c)
- Are there any findings the agent missed that should be investigated?

**Gate 2: After first Confirmation round**
The agent shows what downstream checks found. Human confirms the framing:
- The CXCL12 framing change: "not Areg-specific, reframe as tissue-wide switch"
- The Treg result: "not a failure, report as unexpected finding with interpretation"

**Gate 3: Before Narrative**
The agent shows the evidence scorecard with proposed strength ratings. Human adjusts
framing before the document is written.

The pipeline between gates can run fully autonomously. The gates are cheap — each
requires one human decision, not sustained attention.

---

## Why this isn't a generic deep researcher

The most important structural difference is the **two data-execution loops** that a
generic agent cannot perform:

```
Generic agent:    query → web_search → synthesize → report
                  (grounded by citations)

This pipeline:    data → execute → credibility_check(data) 
                                        ↓
                               novelty_check(web)
                                        ↓
                               confirm(data)
                                        ↓
                                 narrative
                  (grounded by numbers from code on the actual dataset)
```

A generic deep researcher can find that Fndc5 is associated with irisin signaling.
It cannot tell you whether Fndc5 is upregulated in Areg cells specifically, whether
that result holds in every individual mouse, whether it's absent in muscle FAPs, or
whether Il33 is co-expressed in the same cells. Those claims require data execution.

The other structural difference is the **context document as accumulated domain knowledge**.
The LLM doesn't need pre-trained biology to generate good hypotheses — it needs to read
the context document, which encodes what was found, what was ruled out, and what the
literature said. This means the system gets smarter as it runs, not because the model
changes but because the context grows.

---

## Generalizing beyond this dataset

The pipeline is dataset-agnostic. The entry point changes based on the gap identified:

| Gap type | Scanner Agent runs | Credibility checks |
|----------|-------------------|--------------------|
| Better DEG method | Pseudobulk DESeq2 | Rank separation, cell-state specificity, cross-tissue |
| Missing CCC analysis | Manual LR scoring or LIANA | LR score specificity, receptor co-expression |
| Missing trajectory | DPT/Monocle | Pseudotime correlation, lineage marker enrichment |
| Missing cross-tissue | Atlas-wide gene expression | Direction consistency, cell-type equivalence |
| Missing rescue contrast | DESeq2 TH vs SH | Per-mouse separation, overlap with obesity DEGs |

The Hypothesis Agent rules (receptor check, cleavage machinery, co-expression, cross-tissue
equivalence, downstream pathway, bi-directional regulation, honest negative) apply to any
scRNA-seq finding regardless of which gap produced it.

---

## Implementation starting point

The fastest path to a working prototype:

1. **Build the context document writer** — a function that formats a `Finding` into the
   PIPELINE.md schema and appends it. This already exists in spirit (you updated PIPELINE.md
   manually after each notebook); just automate the writes.

2. **Build the three credibility check functions** — `check_rank_separation`,
   `check_cell_state_specificity`, `check_cross_tissue_direction` are fully coded above.
   Wrap them in a `CredibilityAgent` class that takes a `Finding` and returns a
   `CredibilityResult`.

3. **Build the novelty query constructor** — `build_novelty_query` is coded above.
   Connect it to a web search tool and add the two validation rules.

4. **Use the Hypothesis Agent system prompt as-is** — it's rules, not biology.
   The LLM fills in the biology from reading the context document.

5. **Add the approval gates as Slack/email notifications** — the pipeline writes a
   checkpoint JSON, sends a message, polls for a reply. One file + one webhook.

The Scanner Agent (running DESeq2 at scale) and the Narrative Agent (writing the notebook)
are the most complex to build but also the most mechanical — they require reliable code
execution, not novel reasoning.